# 03 — Sentiment Analysis Results

Phân tích kết quả Task 1 (MSA) từ `experiments/results/`:
1. Training curves (loss / accuracy per epoch)
2. So sánh quantum vs classical (E1 vs E2-E4)
3. Confusion matrix trên test set
4. Bảng metrics tổng hợp cho paper

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "experiments" / "results"
FIG_DIR = PROJECT_ROOT / "paper" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load experiment results

In [ ]:
def load_json(path):
    p = Path(path)
    if not p.exists():
        print(f"[missing] {p}")
        return None
    with open(p) as f:
        return json.load(f)


results_all = load_json(RESULTS_DIR / "results.json") or {}
sentiment_runs = load_json(RESULTS_DIR / "sentiment_results.json") or {}

# Gộp các run sentiment từ cả hai nguồn
runs = {**sentiment_runs}
for key in ["E_qfl_tensor", "E_qfl_attention", "E_qfl_interference", "E5_multi_task"]:
    if key in results_all and isinstance(results_all[key], dict) and "history" in results_all[key]:
        runs[key] = results_all[key]

print("Available runs:", list(runs.keys()))

## 2. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, r in runs.items():
    hist = r.get("history", [])
    if not hist:
        continue
    epochs = [h["epoch"] for h in hist]
    axes[0].plot(epochs, [h["train_loss"] for h in hist], label=name)
    if "val_accuracy" in hist[0]:
        axes[1].plot(epochs, [h.get("val_accuracy") for h in hist], label=name)
    elif "val_f1_macro" in hist[0]:
        axes[1].plot(epochs, [h.get("val_f1_macro") for h in hist], label=name)

axes[0].set_title("Training loss")
axes[0].set_xlabel("epoch")
axes[0].set_yscale("log")
axes[1].set_title("Validation metric")
axes[1].set_xlabel("epoch")
axes[0].legend(fontsize=8)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "sentiment_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Quantum vs Classical comparison

In [ ]:
rows = []
for name, r in runs.items():
    test = r.get("test", {})
    rows.append({
        "run": name,
        "accuracy": test.get("accuracy"),
        "f1_macro": test.get("f1_macro"),
        "f1_weighted": test.get("f1_weighted"),
        "mae": test.get("mae"),
        "pearson_r": test.get("pearson_r"),
        "quantum_params": (r.get("params") or {}).get("quantum"),
        "total_params": (r.get("params") or {}).get("total"),
    })

df = pd.DataFrame(rows).set_index("run").sort_values("accuracy", ascending=False)
df.round(4)

In [ ]:
if len(df):
    fig, ax = plt.subplots(figsize=(10, max(3, 0.5 * len(df))))
    colors = ["#2196F3" if ("qfl" in i or "quantum" in i or "qmmf" in i) else "#FF9800" for i in df.index]
    df["accuracy"].plot.barh(ax=ax, color=colors)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Test Accuracy")
    ax.set_title("Sentiment Analysis: Test Accuracy by Model")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "sentiment_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Chưa có kết quả — chạy experiments/run_sentiment.py trước.")

## 4. Confusion matrix (best run)

In [ ]:
# Tìm run tốt nhất có confusion matrix
cm_data = None
best_name = None
best_acc = -1.0
for name, r in runs.items():
    cm = (r.get("test") or {}).get("confusion_matrix")
    acc = ((r.get("test") or {}).get("accuracy")) or 0
    if cm and acc > best_acc:
        cm_data = np.array(cm)
        best_name = name
        best_acc = acc

if cm_data is not None:
    labels = ["positive", "negative", "neutral"]
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_data, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix — {best_name}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Chưa có confusion matrix trong results.")

## 5. Export LaTeX table cho paper

In [ ]:
def to_latex_row(name, row):
    def f(v, kind=".4f"):
        return f"{v:{kind}}" if isinstance(v, (int, float)) else "--"
    pretty = name.replace("_", "\\_")
    return (f"{pretty} & {f(row['accuracy'])} & {f(row['f1_macro'])} & "
            f"{f(row['mae'])} \\\\")

if len(df):
    lines = [
        "\\begin{table}[t]",
        "\\centering",
        "\\begin{tabular}{lccc}",
        "\\toprule",
        "Model & Accuracy & F1 (macro) & MAE \\\\",
        "\\midrule",
    ]
    lines += [to_latex_row(n, r) for n, r in df.iterrows()]
    lines += ["\\bottomrule", "\\end{tabular}",
              "\\caption{Sentiment analysis results on MVSA-Single test set.}",
              "\\label{tab:msa_results}", "\\end{table}"]
    out = FIG_DIR / "msa_table.tex"
    out.write_text("\n".join(lines))
    print("Saved:", out)
    print("\n".join(lines))